In [1]:
import random
import anthropic
import wandb
import pandas as pd
from datasets import load_dataset
from pathlib import Path

/Users/kento/Desktop/Projects/nlp/text-to-sql/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv('raw/IT_Support_Ticket Data.csv')
print(f'Dataset cargado:\n{len(df)} Tickets\n{len(df.columns)} Columnas')

Dataset cargado:
29651 Tickets
5 Columnas


In [16]:
df.head()

,Unnamed: 0,Body,Department,Priority,Tags
0,0,"Dear Customer Support Team,I am writing to rep...",Technical Support,high,"['Account', 'Disruption', 'Outage', 'IT', 'Tec..."
1,1,"Dear Customer Support Team,I hope this message...",Returns and Exchanges,medium,"['Product', 'Feature', 'Tech Support']"
2,2,"Dear Customer Support Team,I hope this message...",Billing and Payments,low,"['Billing', 'Payment', 'Account', 'Documentati..."
3,3,"Dear Support Team,I hope this message reaches ...",Sales and Pre-Sales,medium,"['Product', 'Feature', 'Feedback', 'Tech Suppo..."
4,4,"Dear Customer Support,I hope this message reac...",Technical Support,high,"['Feature', 'Product', 'Documentation', 'Feedb..."


In [17]:
df.drop(columns=['Unnamed: 0'])

,Body,Department,Priority,Tags
0,"Dear Customer Support Team,I am writing to rep...",Technical Support,high,"['Account', 'Disruption', 'Outage', 'IT', 'Tec..."
1,"Dear Customer Support Team,I hope this message...",Returns and Exchanges,medium,"['Product', 'Feature', 'Tech Support']"
2,"Dear Customer Support Team,I hope this message...",Billing and Payments,low,"['Billing', 'Payment', 'Account', 'Documentati..."
3,"Dear Support Team,I hope this message reaches ...",Sales and Pre-Sales,medium,"['Product', 'Feature', 'Feedback', 'Tech Suppo..."
4,"Dear Customer Support,I hope this message reac...",Technical Support,high,"['Feature', 'Product', 'Documentation', 'Feedb..."
...,...,...,...,...
29646,"Hello Customer Support Team,My name is name. I...",IT Support,high,"['Technical Support', 'Urgent Issue', 'Service..."
29647,"Dear Customer Support Team,We are currently en...",IT Support,high,"['Service Disruption', 'IT Support', 'Urgent I..."
29648,"Dear Customer Support Team,Our Cisco Router IS...",Technical Support,high,"['Technical Support', 'Network Issue', 'Urgent..."
29649,"Dear IT Services Support Team, I am writing to...",Technical Support,high,"['Urgent Issue', 'Service Disruption', 'Incide..."


In [4]:
df['Department'].unique()

<ArrowStringArray>
[              'Technical Support',           'Returns and Exchanges',
            'Billing and Payments',             'Sales and Pre-Sales',
 'Service Outages and Maintenance',                 'Product Support',
                      'IT Support',                'Customer Service',
                 'Human Resources',                 'General Inquiry']
Length: 10, dtype: str

Creamos categorías más genéricas y el equipo encargado de la gestión

In [21]:
CATEGORIES = ['biling', 'technical', 'shipping', 'account', 'general']
URGENCY = ['low', 'medium', 'hgih', 'critical']
TEAMS = ['biling-team', 'tech-support', 'logistics', 'account-team', 'general-support']

In [22]:
#Map del df a nuestras categorías y equipos

DEPARTMENT_TO_CATEGORY = {
    'Technical Support': 'technical',
    'Billing and Payments': 'biling',
    'Service Outages and Maintenance': 'account',
    'IT Support': 'technical',
    'Human Resources': 'general',
    'Returns and Exchanges': 'shipping',
    'Sales and Pre-Sales': 'general',
    'Product Support': 'technical',
    'Customer Service': 'account',
    'General Inquiry': 'general'
}

CATEGORY_TO_TEAM = {
    'technical': 'tech-support',
    'biling': 'biling-team',
    'account': 'account-team',
    'general': 'general-support',
    'shipping': 'logistics'
}

In [26]:
def process_real_tickets(df: pd.DataFrame) -> list[dict]:
    processed = []
    for idx, row in df.iterrows():
        department = str(row.get('Department', 'None'))
        category = DEPARTMENT_TO_CATEGORY.get(department, 'general')
        urgency = str(row.get('Priority', 'None'))
        team = CATEGORY_TO_TEAM.get(category, 'general-support')
        ticket_text = str(row.get('Body', ''))
        if not ticket_text or len(ticket_text) < 5:
            continue
        processed.append({
            'text': ticket_text,
            'category': category,
            'urgency': urgency,
            'team': team
        })
    print(f'Processed {len(processed)} real tickets')
    return processed

In [57]:
def find_underrepresented_cases(dataset: list[dict]) -> list[tuple]:
    min_count = 1500
    counts = {}
    for ticket in dataset:
        key = (ticket['category'], ticket['urgency'])
        counts[key] = counts.get(key, 0) +1
    underrepresented = [key for key, count in counts.items() if count < min_count]
    return underrepresented

In [27]:
tickets_real = process_real_tickets(df)

Processed 29649 real tickets


In [58]:
underrepresented = find_underrepresented_cases(tickets_real)

In [59]:
underrepresented

[('shipping', 'medium'),
 ('biling', 'low'),
 ('general', 'medium'),
 ('general', 'high'),
 ('biling', 'medium'),
 ('shipping', 'high'),
 ('biling', 'high'),
 ('shipping', 'low'),
 ('general', 'low')]

In [ ]:
def label_ticket(text:str) -> dict:
    prompt = f"""Clasifica este ticket de soporte al cliente.
Devuelve solo JSON, sin texto adicional:
{{
  "category": {CATEGORIES},
  "urgency": {URGENCY},
  "team": {TEAMS},
  "reasoning": "una frase explicando la clasificación"
}}

Ticket: {text[:500]}"""

    response = client.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=200,
        messages=[{'role':'user', 'content':prompt}]
    )
    return json.loads(response.content[0].text)

def load_and_label_real(n_samples: int = 3000) -> list[dict]:
    dataset = load_dataset('twcs', split='train')

In [17]:
dataset = load_dataset("tweet_eval", split="train")

ValueError: Config name is missing.
Please pick one among the available configs: ['emoji', 'emotion', 'hate', 'irony', 'offensive', 'sentiment', 'stance_abortion', 'stance_atheism', 'stance_climate', 'stance_feminist', 'stance_hillary']
Example of usage:
	`load_dataset('tweet_eval', 'emoji')`

In [12]:
dataset[100]

{'text': 'can you share card tracking number?', 'label': 11}